# Velora — Build + Simulation
Upload the GitHub repository ZIP. This notebook builds `Velora-debug.apk` and, when the Colab runtime exposes KVM, launches a headless Android emulator and captures the main Velora Simulation Mode screens.

## 1. Upload the GitHub ZIP
On GitHub choose **Code → Download ZIP**, then run the next cell and upload that ZIP.

In [ ]:
from google.colab import files
uploaded = files.upload()
zip_name = next(name for name in uploaded if name.lower().endswith('.zip'))
print('Using:', zip_name)

## 2. Extract and detect the Android project

In [ ]:
import os, pathlib, shutil, subprocess, zipfile, time
work = pathlib.Path('/content/velora-build')
if work.exists(): shutil.rmtree(work)
work.mkdir(parents=True)
with zipfile.ZipFile('/content/' + zip_name) as z: z.extractall(work)
candidates = list(work.rglob('settings.gradle.kts')) + list(work.rglob('settings.gradle'))
if not candidates: raise RuntimeError('No Gradle Android project found in ZIP')
project = min((p.parent for p in candidates), key=lambda p: len(p.parts))
print('Project root:', project)

## 3. Install JDK + Android SDK

In [ ]:
import os, pathlib, shutil, subprocess
subprocess.run('apt-get update -qq', shell=True, check=True)
subprocess.run('DEBIAN_FRONTEND=noninteractive apt-get install -y -qq openjdk-17-jdk wget unzip ca-certificates', shell=True, check=True)

sdk = pathlib.Path('/content/android-sdk')
cmdline = sdk / 'cmdline-tools' / 'latest'
sdkmanager_path = cmdline / 'bin' / 'sdkmanager'

if not sdkmanager_path.exists():
    archive = pathlib.Path('/content/cmdline-tools.zip')
    temp = pathlib.Path('/content/cmdline-temp')
    subprocess.run(['wget','-q','https://dl.google.com/android/repository/commandlinetools-linux-11076708_latest.zip','-O',str(archive)], check=True)
    if temp.exists(): shutil.rmtree(temp)
    temp.mkdir(parents=True)
    subprocess.run(['unzip','-q',str(archive),'-d',str(temp)], check=True)
    cmdline.parent.mkdir(parents=True, exist_ok=True)
    if cmdline.exists(): shutil.rmtree(cmdline)
    shutil.move(str(temp / 'cmdline-tools'), str(cmdline))
    shutil.rmtree(temp, ignore_errors=True)

os.environ['ANDROID_SDK_ROOT'] = str(sdk)
os.environ['ANDROID_HOME'] = str(sdk)
os.environ['PATH'] = str(cmdline / 'bin') + ':' + str(sdk / 'platform-tools') + ':' + str(sdk / 'emulator') + ':' + os.environ['PATH']

sdkmanager = shutil.which('sdkmanager') or str(sdkmanager_path)
if not pathlib.Path(sdkmanager).exists():
    raise RuntimeError('sdkmanager was not installed correctly')

pathlib.Path(sdkmanager).chmod(pathlib.Path(sdkmanager).stat().st_mode | 0o111)
subprocess.run('yes | sdkmanager --licenses >/dev/null', shell=True, check=False)
subprocess.run(['sdkmanager','platform-tools','platforms;android-35','build-tools;35.0.0'], check=True)

(project / 'local.properties').write_text('sdk.dir=' + str(sdk) + '\n')
print('Android SDK ready:', sdk)
subprocess.run('sdkmanager --list_installed | grep -E "platform-tools|android-35|35.0.0"', shell=True, check=False)


## 4. Build Velora APK

In [ ]:
import hashlib, json, os, pathlib, shutil, subprocess, time

gradle_home = pathlib.Path('/content/gradle-8.9')
if not gradle_home.exists():
    gradle_zip = pathlib.Path('/content/gradle-8.9-bin.zip')
    subprocess.run(['wget','-q','https://services.gradle.org/distributions/gradle-8.9-bin.zip','-O',str(gradle_zip)], check=True)
    subprocess.run(['unzip','-q',str(gradle_zip),'-d','/content'], check=True)

gradle = gradle_home / 'bin' / 'gradle'
if not gradle.exists():
    raise RuntimeError('Gradle 8.9 was not installed correctly')
gradle.chmod(gradle.stat().st_mode | 0o111)

env = os.environ.copy()
env['JAVA_HOME'] = '/usr/lib/jvm/java-17-openjdk-amd64'
env['ANDROID_HOME'] = '/content/android-sdk'
env['ANDROID_SDK_ROOT'] = '/content/android-sdk'
env['PATH'] = '/content/android-sdk/cmdline-tools/latest/bin:/content/android-sdk/platform-tools:' + env['PATH']

print('Building Velora debug APK...')
print('Project:', project)
process = subprocess.Popen(
    [str(gradle), '--no-daemon', '--stacktrace', ':app:assembleDebug'],
    cwd=project,
    env=env,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)
build_lines = []
for line in process.stdout:
    print(line, end='')
    build_lines.append(line)
exit_code = process.wait()
print('\nEXIT CODE:', exit_code)
if exit_code != 0:
    print('\n--- LAST BUILD LINES ---')
    print(''.join(build_lines[-160:]))
    raise RuntimeError('Velora build failed. See the Gradle output above.')

apks = sorted((project / 'app' / 'build' / 'outputs' / 'apk' / 'debug').glob('*.apk'), key=lambda p: p.stat().st_mtime)
if not apks:
    raise RuntimeError('Build succeeded but APK was not found')

output = pathlib.Path('/content/Velora-debug.apk')
shutil.copy2(apks[-1], output)
sha256 = hashlib.sha256(output.read_bytes()).hexdigest()
checksum = pathlib.Path('/content/Velora-debug.apk.sha256')
checksum.write_text(sha256 + '  Velora-debug.apk\n')

metadata = {
    'artifact': output.name,
    'size_bytes': output.stat().st_size,
    'sha256': sha256,
    'variant': 'debug',
    'android_api': 35,
    'build_tools': '35.0.0',
    'gradle': '8.9',
    'java': '17',
}
metadata_path = pathlib.Path('/content/Velora-build.json')
metadata_path.write_text(json.dumps(metadata, indent=2))

apksigner = pathlib.Path('/content/android-sdk/build-tools/35.0.0/apksigner')
if apksigner.exists():
    apksigner.chmod(apksigner.stat().st_mode | 0o111)
    print('\nAPK signature verification:')
    subprocess.run([str(apksigner), 'verify', '--verbose', '--print-certs', str(output)], check=True)

print('\nVELORA APK BUILD SUCCESSFUL')
print('APK     :', output)
print('Size    :', f'{output.stat().st_size / 1024 / 1024:.2f} MiB')
print('SHA-256 :', sha256)


## 5. Optional local Colab emulator + screenshots
Leave `RUN_LOCAL_EMULATOR = True`. The notebook checks KVM first. If this Colab runtime does not expose KVM, it safely skips the emulator and keeps the APK.

In [ ]:
RUN_LOCAL_EMULATOR = True
preview_zip = None
has_kvm = pathlib.Path('/dev/kvm').exists() and os.access('/dev/kvm', os.R_OK | os.W_OK)
print('KVM available:', has_kvm)

if RUN_LOCAL_EMULATOR and has_kvm:
    system_image = 'system-images;android-35;google_apis;x86_64'
    subprocess.run(['sdkmanager','emulator',system_image], check=True)
    avdmanager = shutil.which('avdmanager') or '/content/android-sdk/cmdline-tools/latest/bin/avdmanager'
    avd_path = pathlib.Path(avdmanager)
    if avd_path.exists(): avd_path.chmod(avd_path.stat().st_mode | 0o111)
    android_home = pathlib.Path('/content/.android')
    android_home.mkdir(exist_ok=True)
    env['ANDROID_AVD_HOME'] = str(android_home / 'avd')
    pathlib.Path(env['ANDROID_AVD_HOME']).mkdir(parents=True, exist_ok=True)
    subprocess.run(
        f'echo no | "{avdmanager}" create avd -n velora_api35 -k "{system_image}" --device "pixel_6" --force',
        shell=True, env=env, check=True
    )
    emulator = pathlib.Path('/content/android-sdk/emulator/emulator')
    emulator.chmod(emulator.stat().st_mode | 0o111)
    log = open('/content/velora-emulator.log', 'w')
    emulator_proc = subprocess.Popen([
        str(emulator), '-avd', 'velora_api35', '-no-window', '-no-audio',
        '-no-boot-anim', '-no-snapshot', '-wipe-data', '-gpu', 'swiftshader_indirect'
    ], stdout=log, stderr=subprocess.STDOUT, env=env)
    adb = pathlib.Path('/content/android-sdk/platform-tools/adb')
    adb.chmod(adb.stat().st_mode | 0o111)
    subprocess.run([str(adb), 'wait-for-device'], check=True)
    booted = False
    for _ in range(180):
        state = subprocess.run([str(adb), 'shell', 'getprop', 'sys.boot_completed'], capture_output=True, text=True).stdout.strip()
        if state == '1':
            booted = True
            break
        time.sleep(2)
    if not booted:
        raise RuntimeError('Emulator did not finish booting. See /content/velora-emulator.log')
    subprocess.run([str(adb), 'install', '-r', str(output)], check=True)
    preview_dir = pathlib.Path('/content/velora-previews')
    if preview_dir.exists(): shutil.rmtree(preview_dir)
    preview_dir.mkdir()
    for screen in ['home','drawer','control','settings','widgets']:
        subprocess.run([str(adb),'shell','am','force-stop','tech.wonderer.velora'], check=False)
        subprocess.run([str(adb),'shell','am','start','-W','-n','tech.wonderer.velora/.SimulationActivity','--es','screen',screen], check=True, stdout=subprocess.DEVNULL)
        time.sleep(2)
        with open(preview_dir / f'velora-{screen}.png', 'wb') as image_file:
            subprocess.run([str(adb),'exec-out','screencap','-p'], stdout=image_file, check=True)
    preview_zip = pathlib.Path(shutil.make_archive('/content/Velora-previews','zip',preview_dir))
    print('Preview screenshots:', preview_zip)
else:
    print('Local emulator skipped. APK build is still valid; use the Velora Preview entry on a browser/cloud emulator or phone.')


## 6. Download outputs
The APK always downloads. If the local emulator ran, the screenshot ZIP downloads too.

In [ ]:
from google.colab import files
files.download('/content/Velora-debug.apk')
files.download('/content/Velora-debug.apk.sha256')
files.download('/content/Velora-build.json')
if preview_zip is not None and pathlib.Path(preview_zip).exists():
    files.download(str(preview_zip))
